# Python client for DB Service

The DB Service Python client provides a thin wrapper over the DB Service APIs, making it easier to connect to a running service and perform client operations from Python.

Use this client to create a session, run queries, manage tables, and interact with DB Service from Python.

## Connect

Create a session to connect to DB Service:

In [1]:
import dbservice_client as dbs
import pandas as pd  # Query examples return dataframes
from datetime import datetime as dt, timezone, timedelta

# Default endpoint: localhost:8080
session = dbs.Session()

# Optional explicit endpoint
rest_session = dbs.Session(endpoint="localhost:8080")


Welcome to KDB-X Community Edition!
For Community support, please visit https://kx.com/slack
Tutorials can be found at https://github.com/KxSystems/tutorials
Ready to go beyond the Community Edition? Email preview@kx.com



## Managing Tables
Use these calls to define and inspect table schemas in DB Service.

In [2]:
# List tables (empty to begin with)
session.list_tables()

[]

In [3]:
# Create partitioned table ('fxquote')
session.create_table(
    table="fxquote",
    type="partitioned",
    prtnCol="ts",
    sortColsDisk=["sym"],
    sortColsOrd=["sym"],
    columns=[
        {"name": "trddate", "type": "date"},
        {"name": "ts", "type": "timestamp"},
        {"name": "sym", "type": "symbol", "attrMem": "grouped", "attrDisk": "parted", "attrOrd": "parted"},
        {"name": "bid", "type": "float"},
        {"name": "ask", "type": "float"},
    ]
)

{'name': 'dc8cc00c-2f34-0199-604f-9a58551c794d',
 'pipeline': '',
 'database': 'db',
 'updtype': 'schemaChange',
 'status': 'completed',
 'details': [],
 'tbls': [],
 'dates': [],
 'progress': {'cmdCurrent': '',
  'cmdIndex': 0,
  'cmdTotal': 0,
  'subCurrent': '',
  'subIndex': 0,
  'subTotal': 0},
 'error': '',
 'warnings': [],
 'updated': '2026-04-27T11:14:19.947873238'}

In [4]:
# List tables ('fxquote' table returned)
session.list_tables()

['fxquote']

In [5]:
# Describe the 'fxquote' table
session.describe_table(table="fxquote")

{'columns': [{'name': 'trddate', 'type': 'date'},
  {'name': 'ts', 'type': 'timestamp'},
  {'name': 'sym',
   'type': 'symbol',
   'attrMem': 'grouped',
   'attrDisk': 'parted',
   'attrOrd': 'parted'},
  {'name': 'bid', 'type': 'float'},
  {'name': 'ask', 'type': 'float'}],
 'type': 'partitioned',
 'prtnCol': 'ts',
 'sortColsOrd': ['sym'],
 'sortColsDisk': ['sym'],
 'name': 'fxquote'}

## Importing Data
DB Service supports both `file-based` and `in-memory` ingest. Any file you want to import must first be copied into the DB Service `imports` staging directory, for example: `~/.kx/db-service/data/imports/`

### Import CSV

In [6]:
# Import a CSV file into the existing 'fxquote' table
job = session.import_files(table="fxquote", path="fxquote.csv.gz")

In [7]:
# Check the status of the above import job
session.get_import(job_id=job["name"])

{'name': 'ee32cc73-493e-ce53-cd7a-d0802b7f9604',
 'pipeline': '',
 'database': 'db',
 'updtype': 'ingest',
 'status': 'completed',
 'details': {'preprocessDone': '2026-04-27T11:14:54.895524326',
  'subsessions': [''],
  'dates': ['2026-03-02'],
  'tables': ['fxquote']},
 'tbls': ['fxquote'],
 'dates': ['2026-03-02'],
 'progress': {'cmdCurrent': '',
  'cmdIndex': 1,
  'cmdTotal': 1,
  'subCurrent': '',
  'subIndex': 0,
  'subTotal': 0},
 'error': '',
 'warnings': [],
 'updated': '2026-04-27T11:14:55.129310425'}

In [8]:
# Import a parquet file into the existing 'fxquote' table
job = session.import_files(table='fxquote', path='fxquote.parquet')

In [9]:
# Import a CSV file and create the 'instruments' table automatically if it does not exist
job = session.import_files(table="instruments", path="instruments.csv", createTable=True)

### Import JSON
Users can import data directly from Python without file staging.

In [10]:
# Objects payload imported to 'instruments' table
job = session.import_data(
    table="instruments",
    data=[
        {"instrumentid": 77, "sym": "USDBRL", "category": "EM", "decimals": 4, "pipdecimals": 4},
        {"instrumentid": 78, "sym": "USDKRW", "category": "EM", "decimals": 2, "pipdecimals": 2},
    ],
    insert_as="objects",
)

In [11]:
# Rows payload imported to the 'fxquote' table
job = session.import_data(
    table="fxquote",
    data=[
        ["2026-01-21", "2026-01-21T10:00:00.000", "EURUSD", 901.2, 901.3],
        ["2026-01-21", "2026-01-21T10:00:00.000", "EURUSD", 901.2, 901.3],
    ],
    columnNames=["trddate", "ts", "sym", "bid", "ask"],
    insert_as="rows",
)
# Note: for rows payload, columnNames are required.

{'name': 'bacbd476-9299-b174-9421-a1af6e20dcda',
 'pipeline': '',
 'database': 'db',
 'updtype': 'ingest',
 'status': 'pending',
 'details': [],
 'tbls': [],
 'dates': [],
 'progress': {'cmdCurrent': '',
  'cmdIndex': None,
  'cmdTotal': None,
  'subCurrent': '',
  'subIndex': None,
  'subTotal': None},
 'error': '',
 'warnings': [],
 'updated': '2026-04-27T11:15:19.863698088'}

## Querying Tables
Run structured, SQL, or q queries against DB Service.

In [12]:
# Structured query
session.query_simple(
    table="fxquote",
    startTS="2026.03.02D00:00:00.000",
    endTS="2026.03.03D00:00:00.000",
    sortCols=["ts"],
    limit=5,
    return_as="json",
)

[{'trddate': '2026-03-02',
  'ts': '2026-03-02T00:00:00.000000000',
  'sym': 'AUDUSD',
  'bid': 0.67091,
  'ask': 0.67094},
 {'trddate': '2026-03-02',
  'ts': '2026-03-02T00:00:00.000000000',
  'sym': 'EURUSD',
  'bid': 1.16397,
  'ask': 1.16399},
 {'trddate': '2026-03-02',
  'ts': '2026-03-02T00:00:00.000000000',
  'sym': 'GBPUSD',
  'bid': 1.3419,
  'ask': 1.34194},
 {'trddate': '2026-03-02',
  'ts': '2026-03-02T00:00:00.000000000',
  'sym': 'USDCAD',
  'bid': 1.38744,
  'ask': 1.3875},
 {'trddate': '2026-03-02',
  'ts': '2026-03-02T00:00:00.000000000',
  'sym': 'USDJPY',
  'bid': 158.162,
  'ask': 158.167}]

In [13]:
# SQL query
session.query_sql(
    query="SELECT * FROM instruments WHERE category LIKE 'EM'",
    return_as="pandas",
)

,instrumentid,sym,category,decimals,pipdecimals
0,14,CHFZAR,EM,5,4
1,29,EURTRY,EM,5,4
2,31,EURZAR,EM,5,4
3,41,GBPZAR,EM,5,4
4,61,USDCNH,EM,5,4
5,66,USDINR,EM,5,4
6,68,USDMXN,EM,5,4
7,73,USDTHB,EM,3,2
8,74,USDTRY,EM,5,4
9,75,USDZAR,EM,5,4


In [14]:
# QSQL query
session.query_q(
    query='select o:first bid,h:max bid,l:min bid,c:last bid by trddate,sym from fxquote',
    return_as="pandas",
)

,trddate,sym,o,h,l,c
0,2026-01-21,EURUSD,901.20000,901.20000,901.20000,901.20000
1,2026-03-02,AUDUSD,0.67091,0.67469,0.67065,0.67307
2,2026-03-02,EURUSD,1.16397,1.17680,1.16325,1.17268
3,2026-03-02,GBPUSD,1.34190,1.34913,1.34101,1.34404
4,2026-03-02,USDCAD,1.38744,1.38790,1.38140,1.38336
5,2026-03-02,USDJPY,158.16200,158.60200,157.47600,158.15100
6,2026-03-03,AUDUSD,0.67310,0.67781,0.67273,0.67540
7,2026-03-03,EURUSD,1.17258,1.17430,1.16703,1.16726
8,2026-03-03,GBPUSD,1.34401,1.34588,1.33996,1.34176
9,2026-03-03,USDCAD,1.38338,1.38441,1.37855,1.38440


**Return format:** `return_as` may be `json`, `pandas`, or `pykx`. If omitted, it defaults to `json`.

## Deleting tables

In [15]:
# List tables (expected: 'fxquote' and 'instruments')
session.list_tables()

['fxquote', 'instruments']

In [16]:
# Drop the 'instruments' table
session.drop_table(table="instruments")

{'name': '4749299e-f17f-7bfb-70f5-aaae237696d1',
 'pipeline': '',
 'database': 'db',
 'updtype': 'schemaChange',
 'status': 'completed',
 'details': [],
 'tbls': [],
 'dates': [],
 'progress': {'cmdCurrent': '',
  'cmdIndex': 0,
  'cmdTotal': 0,
  'subCurrent': '',
  'subIndex': 0,
  'subTotal': 0},
 'error': '',
 'warnings': [],
 'updated': '2026-04-27T11:16:04.554099413'}

In [17]:
# List tables (expected: instruments is gone)
session.list_tables()

['fxquote']